# Produce various [El1/El2] vs. Age plots for abritrary-ish element ratios

This should definitely plot the abundances versus age for the Bensby samples so I can see how the element ratios evolve over time relative to each other instead of just relative to Iron.

It's decently possible I'll end up making this code also plot 2 element ratios arbitarily for the stars.... but then that's what the other code does it's just not solar-normalized... so I suppose it's conceivable those should be overlapping codes... hmmmm. In any case, this will pull heavily from generate_figure3.ipynb (the log(Li/Ca) vs. Age plot from the Science paper).

In [1]:


from __future__ import print_function


import matplotlib
matplotlib.use('pdf')


import numpy as np
import matplotlib.pyplot as plt
import sys
import os
from astropy.io import fits
from glob import glob
from astropy.time import Time
from astropy import coordinates as coords
from astropy import units as u
from astropy import constants as const
from astropy import convolution as conv
from astropy.table import Table, Column
import scipy.interpolate as scinterp
import time
start = time.time()

import spec_plot_tools as spt
import cal_params as cp
import plot_spec as ps
import bensby_plotting as bp
import abundance_corrections as acorr
import interp_tau as itau
import fix_strings as fs




print(os.getcwd())

No handles with labels found to put in legend.


all_avg
(116, 4, 27)
(4, 27, 116)
(116,)
(116,)
(27, 116)
(116,)
(116,)
(27, 116)
(116,)
(116,)
(27, 116)
(116,)
(116,)
(27, 116)
/Users/BenKaiser/Desktop/radial_velocity_calculations


/Users/BenKaiser/Desktop/radial_velocity_calculations/interp_tau.py:62: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [2]:
figure_output_dir='/Users/BenKaiser/Desktop/'

target_dir= '/Users/BenKaiser/Desktop/radial_velocity_calculations/'
os.chdir(target_dir)

In [3]:
#wd_abund_file='all_wd_abundances.csv'
#wd_abund_file='20210131_all_wd_abundances.csv'
#wd_abund_file='20210713_all_wd_abundances_bedard_cooling_no_WDs_plot.csv'
wd_abund_file='20210818_all_wd_abundances_bedard_cooling_newer_Blouin.csv'
lodders_abund_file='Lodders2009_solarsystem_abundances.csv'
#solar_system_object_file='solar_system_body_abundances.csv'
#solar_system_object_file='solar_system_body_abundances_mgfe_fixed.csv'
#solar_system_object_file='solar_system_body_abundances_sea_added.csv'
#solar_system_object_file='20210713_solar_system_body_abundances_show_name_added.csv'
solar_system_object_file='20210713_solar_system_body_abundances_all_names.csv'

In [4]:
wd_abund_table=Table.read(wd_abund_file)
wd_abund_table=spt.clean_color_string(wd_abund_table,color_header='plot_color')
lodders_table=Table.read(lodders_abund_file)
bodies_table=Table.read(solar_system_object_file)
lodders_table.add_index('element')
wd_abund_table.add_index('name')
bodies_table.add_index('name')

limit_length=0.3 #length of limit error bars on plots
limit_indicator=99. #value above which if the absolute value of the error on a measurement is above it indicates it should be a limit



In [19]:
color_dict={
    'WDJ1644-0449':'#ff0000',
    'SDSSJ1330+6435':'#8900ff',
    'WDJ2356-209':'#00ffc5',
    'SDSSJ1636+1619':'pink',
    'WDJ2317+1830':'orange',
    'WDJ1824+1213':'g',
    'LHS2534':'b'
}
step_dict={
    'WDJ1644-0449':5,
    'SDSSJ1330+6435':5,
    'WDJ2356-209':5,
    'SDSSJ1636+1619':5,
    'WDJ2317+1830':5,
    'WDJ1824+1213':5,
    'LHS2534':5
}

wd_marker='*'
#met_marker='D'
#ssp_marker='met_marker'
met_marker='s'
ssp_marker='D'
met_color='#1ca1f2'
#met_color='r'

met_size=3
ci_size=6
wd_size=10
dp_alpha=0.5
ci_leg_size=9
arr_naca=[-0.4,-0.1]
alpha_range=[0.5,0.2]
arrow_segs=100
arrow_width=0.03
#arrow_width=0.07

arrow_line=4
figure_text_size=6
default_offset=[0.05,0.00]
annot_line_weight=0.03

show_all_ssobj_names=False

star_marker='o'
pop_colors=['darkorange','brown','navy','grey'] #thin disk, thick disk, halo, in-between for Bensby plots
#pop_colors=['darkorange','darkorange','darkorange','darkorange'] #for version where we don't distinguish pops



In [24]:
spt.initiate_science_plot()
plt.figure(figsize=(4.75,4.75),constrained_layout=True)

#el1='Mg'

el1='Na'

el2='Ca'

#el1='Ca'
#el2='Mg'

count=0
bp.plot_el1el2_FeH(el1,el2,markersize=4,error_bars=False,alpha=0.2)
plt.errorbar(0, 0, color=met_color, marker=met_marker, markersize=ci_size, linestyle='None')


#bp.plot_el1el2_age_pop(el1, el2, error_bars=True, mask_err_free=True)
#plt.errorbar(4.57, 1.10-6.33, yerr=np.sqrt(0.1**2+0.07**2),marker=star_marker, color=met_color, label="Sun",linestyle='None',markersize=starsize )
#plt.errorbar(4.57,lodders_table.loc['Li']['A_el']-lodders_table.loc['Ca']['A_el'],yerr=np.sqrt(lodders_table.loc['Li']['A_el_err']**2+lodders_table.loc['Ca']['A_el_err']**2),label="CI Chondrites",marker=met_marker, color=met_color, linestyle='None', markersize=ci_leg_size)
#plt.errorbar(4.57,lodders_table.loc['Li']['A_el']-lodders_table.loc['Ca']['A_el'],yerr=np.sqrt(lodders_table.loc['Li']['A_el_err']**2+lodders_table.loc['Ca']['A_el_err']**2),marker=met_marker,markersize=ci_size, color=met_color)
#plt.xlim(14,0)




plt.legend(fontsize=7)
#plt.grid(True)
plt.xlabel('[Fe/H]')
plt.ylabel('['+el1+'/'+el2+']')


print(os.getcwd())
os.chdir(figure_output_dir)
print(os.getcwd())
start = time.time()
print(start)
time_string=str(start).split('.')[0]
plt.savefig(el1+el2+'_v_FeH_Bensby_'+time_string+'.pdf')#plt.grid(True)

plt.show()




No handles with labels found to put in legend.


/Users/BenKaiser/Desktop
/Users/BenKaiser/Desktop
1629315942.4505532


/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:41: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.


In [25]:
spt.initiate_science_plot()
plt.figure(figsize=(4.75,4.75),constrained_layout=True)

#el1='Mg'
#el1='Na'

#el2='Ca'

#el1='Ca'
#el1el2='Na/Mg'
#el3el4='Ca/Mg'

#el1el2='Ca/Fe'
#el3el4='Mg/Fe'

el1el2='Na/Ca'
el3el4='Mg/Fe'


count=0
bp.plot_el1el2_el3el4(el1el2,el3el4,markersize=4,error_bars=False,alpha=0.2)
plt.errorbar(0, 0, color=met_color, marker=met_marker, markersize=ci_size, linestyle='None')
#bp.plot_el1el2_age_pop(el1, el2, error_bars=True, mask_err_free=True)
#plt.errorbar(4.57, 1.10-6.33, yerr=np.sqrt(0.1**2+0.07**2),marker=star_marker, color=met_color, label="Sun",linestyle='None',markersize=starsize )
#plt.errorbar(4.57,lodders_table.loc['Li']['A_el']-lodders_table.loc['Ca']['A_el'],yerr=np.sqrt(lodders_table.loc['Li']['A_el_err']**2+lodders_table.loc['Ca']['A_el_err']**2),label="CI Chondrites",marker=met_marker, color=met_color, linestyle='None', markersize=ci_leg_size)
#plt.errorbar(4.57,lodders_table.loc['Li']['A_el']-lodders_table.loc['Ca']['A_el'],yerr=np.sqrt(lodders_table.loc['Li']['A_el_err']**2+lodders_table.loc['Ca']['A_el_err']**2),marker=met_marker,markersize=ci_size, color=met_color)
#plt.xlim(14,0)




plt.legend(fontsize=7)
#plt.grid(True)
#plt.xlabel('[Fe/H]')
#plt.ylabel('['+el1+'/'+el2+']')


print(os.getcwd())
os.chdir(figure_output_dir)
print(os.getcwd())
start = time.time()
print(start)
time_string=str(start).split('.')[0]
plt.savefig(el1el2.replace('/','')+'_v_'+el3el4.replace('/','')+'_Bensby_'+time_string+'.pdf')#plt.grid(True)

plt.show()





No handles with labels found to put in legend.




****************
KeyError: 'Fe/Fe' 
****************


/Users/BenKaiser/Desktop
/Users/BenKaiser/Desktop
1629315950.877216


/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:46: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.
